In [9]:
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import Bounds
import cma
from numpy.linalg import det
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from itertools import permutations
import seaborn as sns
import os


# Create the output folder on the Desktop
desktop_path = os.path.expanduser("~/Desktop")
output_dir = os.path.join(desktop_path, "lambda_gamma_beta_VAE_results")
os.makedirs(output_dir, exist_ok=True)

# Enable LaTeX rendering for labels (usetex=False to avoid requiring LaTeX installation)
plt.rc('text', usetex=False)

# Set random seed for reproducibility
np.random.seed(42)

# Dimensions and hyperparameters
n = 20  # Data dimension
m = 5   # Latent dimension
s = 5   # Number of generative variables
max_iter = 2000   # Max iterations for optimization
beta_values = [1, 2, 4, 6]  # 4 beta values
lambda_values = [1, 2, 4, 6]  # 4 lambda values
gamma_values = [1, 2, 4, 6]  # 4 gamma values
sigma2 = 0.1  # Noise variance for Sigma_Z_tilde

# Generate Sigma_V (5x5 diagonal matrix for 5 independent generative variables)
sigma_v = 1 + 4 * np.random.rand(s)  # Standard deviations between 1 and 5
Sigma_V = np.diag(sigma_v**2)  # Diagonal matrix of variances

# Generate Gamma (n x s matrix)
Gamma = np.random.randn(n, s)  # Random transformation matrix

# Compute Sigma_Y using the generative model
Sigma_Y = Gamma @ Sigma_V @ Gamma.T + sigma2 * np.eye(n)
# Ensure symmetry and positive definiteness
Sigma_Y = (Sigma_Y + Sigma_Y.T) / 2
eigvals = np.linalg.eigvalsh(Sigma_Y)
if np.any(eigvals <= 0):
    Sigma_Y += (1e-6 - np.min(eigvals)) * np.eye(n)  # Adjust for positive definiteness
Sigma_Y_inv = np.linalg.inv(Sigma_Y)

# Sub-covariance matrix XV
def sub_covariance_XV(B):
    try:
        return B @ Gamma @ Sigma_V  # Shape: (m x s) = (5 x 5)
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Compute mutual information matrix for all pairs (x_i, v_j)
def compute_mutual_information_matrix(B, Sigma_W):
    try:
        # Compute covariance of X: B * Sigma_Y * B.T + Sigma_W
        Sigma_X = B @ Sigma_Y @ B.T + Sigma_W
        if det(Sigma_X) <= 0:
            return np.full((m, s), np.inf)
        
        # Sub-covariance matrix: cov(X, V) = B * Gamma * Sigma_V
        cov_XV = sub_covariance_XV(B)
        if np.any(np.isinf(cov_XV)):
            return np.full((m, s), np.inf)
        
        # Initialize mutual information matrix
        mi_matrix = np.zeros((m, s))
        for i in range(m):
            for j in range(s):
                Sigma_xi = Sigma_X[i, i]  # Variance of x_i
                Sigma_vj = Sigma_V[j, j]  # Variance of v_j
                cov_xi_vj = cov_XV[i, j]  # cov(x_i, v_j)
                Sigma_xi_vj = np.array([[Sigma_xi, cov_xi_vj],
                                       [cov_xi_vj, Sigma_vj]])
                det_xi_vj = det(Sigma_xi_vj)
                if det_xi_vj <= 0:
                    mi_matrix[i, j] = np.inf
                else:
                    mi_matrix[i, j] = 0.5 * np.log(Sigma_xi * Sigma_vj / det_xi_vj)
        return mi_matrix
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Disentanglement metric I_5 (maximum sum over all permutations)
def disentanglement_I5(B, Sigma_W):
    try:
        # Compute the mutual information matrix
        mi_matrix = compute_mutual_information_matrix(B, Sigma_W)
        if np.any(np.isinf(mi_matrix)):
            return np.inf
        
        # Compute I_5 as the maximum sum across all permutations
        max_sum = -np.inf
        for perm in permutations(range(s)):
            current_sum = sum(mi_matrix[i, pi] for i, pi in enumerate(perm))
            max_sum = max(max_sum, current_sum)
        
        return max_sum
    except np.linalg.LinAlgError:
        return np.inf

# Mutual information functions
def encoder_mi(B, Sigma_W):
    try:
        num = B @ Sigma_Y @ B.T + Sigma_W
        det_num = det(num)
        det_den = det(Sigma_W)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

def decoder_mi(A, Sigma_Z):
    try:
        num = A @ A.T + Sigma_Z
        det_num = det(num)
        det_den = det(Sigma_Z)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

# Compute sum of I(X_i; \tilde{X}_i) for each i
def compute_pairwise_mi(B, Sigma_W, A, Sigma_Z):
    try:
        # Compute cov(X) = B * Sigma_Y * B^T + Sigma_W
        Sigma_X = B @ Sigma_Y @ B.T + Sigma_W
        
        # Compute cov(\tilde{X}) = B * A * A^T * B^T + B * Sigma_Z * B^T + Sigma_W
        term1 = B @ (A @ A.T) @ B.T  # B * A * A^T * B^T
        term2 = B @ Sigma_Z @ B.T     # B * Sigma_Z * B^T
        term3 = Sigma_W               # Sigma_W
        Sigma_X_tilde = term1 + term2 + term3
        
        # Initialize sum of I(X_i; \tilde{X}_i)
        mi_sum = 0.0
        for i in range(m):
            # Extract diagonal elements
            sigma_w_ii = Sigma_W[i, i]         # [Sigma_W]_{ii}
            sigma_x_ii = Sigma_X[i, i]         # [B * Sigma_Y * B^T + Sigma_W]_{ii}
            sigma_x_tilde_ii = Sigma_X_tilde[i, i]  # [B * A * A^T * B^T + B * Sigma_Z * B^T + Sigma_W]_{ii}
            
            # Compute the fraction inside the log
            fraction = (sigma_w_ii**2) / (sigma_x_ii * sigma_x_tilde_ii)
            if fraction >= 1 or fraction <= 0:  # Ensure valid log argument
                return np.inf
            
            # Compute I(X_i; \tilde{X}_i) = -0.5 * log(1 - fraction)
            mi_i = -0.5 * np.log(1 - fraction)
            mi_sum += mi_i
        
        return mi_sum  # Return sum for objective function and reporting
    except (np.linalg.LinAlgError, ValueError):
        return np.inf

# Objective function Gamma_1 with reconstruction errors, modified for lambda gamma beta-VAE
def gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_, gamma):
    try:
        Sigma_Z_inv = np.linalg.inv(Sigma_Z)
        Sigma_W_inv = np.linalg.inv(Sigma_W)
        
        # Check determinants for positive definiteness
        det_Sigma_Z = det(Sigma_Z)
        det_Sigma_W = det(Sigma_W)
        if det_Sigma_Z <= 0 or det_Sigma_W <= 0:
            return np.inf, np.inf, np.inf, np.inf
        
        # First term: Log-likelihood reconstruction error
        trace1 = np.trace(A.T @ Sigma_Z_inv @ Sigma_Y @ B.T)
        trace2 = np.trace(Sigma_Z_inv @ A @ B @ Sigma_Y)
        trace3 = np.trace(Sigma_Z_inv @ Sigma_Y)
        trace4 = np.trace(A.T @ Sigma_Z_inv @ A @ (B @ Sigma_Y @ B.T + Sigma_W))
        recon_ll = -0.5 * (trace1 + trace2 - trace3 - trace4 - \
                           n * np.log(2 * np.pi) - np.log(det_Sigma_Z))
        
        # Second term: KL divergence
        term2 = 0.5 * beta * (np.trace(B @ Sigma_Y @ B.T + Sigma_W) - \
                              np.log(det_Sigma_W) - m)
        
        # Third term: L2-norm with lambda for objective
        l2_base = np.trace(
            (np.eye(n) - A @ B) @ Sigma_Y @ (np.eye(n) - A @ B).T + \
            A @ Sigma_W @ A.T
        )
        term3 = lambda_ * l2_base
        recon_l2 = l2_base  # Unscaled L2-norm for reporting
        
        # Compute sum of I(X_i; \tilde{X}_i)
        pairwise_mi_sum = compute_pairwise_mi(B, Sigma_W, A, Sigma_Z)
        if np.isinf(pairwise_mi_sum):
            return np.inf, np.inf, np.inf, np.inf
        
        # New objective: Gamma_1 - gamma * sum I(X_i; \tilde{X}_i)
        gamma_term = gamma * pairwise_mi_sum
        total = recon_ll + term2 + term3 - gamma_term
        
        return total, recon_ll, recon_l2, pairwise_mi_sum
    except np.linalg.LinAlgError:
        return np.inf, np.inf, np.inf, np.inf

# Flatten parameters
def flatten_params(A, B, Sigma_Z, Sigma_W):
    L_W = np.linalg.cholesky(Sigma_W)
    L_W_vec = L_W[np.tril_indices(m)]
    return np.concatenate([
        A.flatten(),
        B.flatten(),
        np.diag(Sigma_Z).flatten(),
        L_W_vec.flatten()
    ])

# Unflatten parameters
def unflatten_params(x, n, m):
    idx_A = n * m
    idx_B = idx_A + m * n
    idx_Sigma_Z = idx_B + n
    A = x[:idx_A].reshape(n, m)
    B = x[idx_A:idx_B].reshape(m, n)
    diag_Z = x[idx_B:idx_Sigma_Z]
    Sigma_Z = np.diag(np.maximum(diag_Z, 1e-1))
    L_W_vec = x[idx_Sigma_Z:]
    L_W = np.zeros((m, m))
    tril_indices = np.tril_indices(m)
    L_W[tril_indices] = L_W_vec
    for i in range(m):
        L_W[i, i] = np.maximum(L_W[i, i], 2)
    Sigma_W = L_W @ L_W.T
    return A, B, Sigma_Z, Sigma_W

# Objective function for L-BFGS-B and CMA-ES
def objective(x, n, m, beta, lambda_, gamma):
    A, B, Sigma_Z, Sigma_W = unflatten_params(x, n, m)
    total, _, _, _ = gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_, gamma)
    return total

# Initial parameters
A_init = 0.1 * np.random.randn(n, m)
B_init = 0.1 * np.random.randn(m, n)
Sigma_Z_init = np.diag(1 + 0.01 * np.abs(np.random.randn(n)))
Sigma_W_init = np.eye(m) + 0.01 * np.random.randn(m, m)
Sigma_W_init = (Sigma_W_init + Sigma_W_init.T) / 2 + 1e-1 * np.eye(m)

# Store results and reconstruction errors for each method
results = []
# Dictionaries to store reconstruction errors for 3D plots
recon_ll_data = {
    'lbfgs': np.zeros((len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.zeros((len(beta_values), len(lambda_values), len(gamma_values)))
}
recon_l2_data = {
    'lbfgs': np.zeros((len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.zeros((len(beta_values), len(lambda_values), len(gamma_values)))
}

# Loop over beta, lambda, and gamma values
for beta_idx, beta in enumerate(beta_values):
    for lambda_idx, lambda_ in enumerate(lambda_values):
        for gamma_idx, gamma in enumerate(gamma_values):
            print(f"\nTesting beta={beta}, lambda={lambda_}, gamma={gamma}")

            # L-BFGS-B Optimization
            x_init = flatten_params(A_init, B_init, Sigma_Z_init, Sigma_W_init)
            lb = np.concatenate([
                -10 * np.ones(n * m),
                -10 * np.ones(m * n),
                1e-1 * np.ones(n),
                -10 * np.ones((m * (m + 1)) // 2)
            ])
            ub = np.concatenate([
                10 * np.ones(n * m),
                10 * np.ones(m * n),
                np.inf * np.ones(n),
                10 * np.ones((m * (m + 1)) // 2)
            ])
            bounds = Bounds(lb, ub)
            result_lbfgs = minimize(
                lambda x: objective(x, n, m, beta, lambda_, gamma), x_init,
                method='L-BFGS-B', bounds=bounds,
                options={'disp': False, 'maxiter': 2000, 'gtol': 1e-5}
            )
            A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs = unflatten_params(result_lbfgs.x, n, m)
            obj_lbfgs, recon_ll_lbfgs, recon_l2_lbfgs, pairwise_mi_lbfgs = gamma_1(A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs, beta, lambda_, gamma)
            mi_enc_lbfgs = encoder_mi(B_lbfgs, Sigma_W_lbfgs)
            mi_dec_lbfgs = decoder_mi(A_lbfgs, Sigma_Z_lbfgs)
            sub_cov_XV_lbfgs = sub_covariance_XV(B_lbfgs)
            mi_matrix_lbfgs = compute_mutual_information_matrix(B_lbfgs, Sigma_W_lbfgs)
            I5_lbfgs = disentanglement_I5(B_lbfgs, Sigma_W_lbfgs)

            # CMA-ES Optimization
            opts = {
                'bounds': [lb, ub],
                'tolfun': 1e-8,
                'maxiter': 2000,
                'maxfevals': 50000,
                'popsize': 100,
                'verb_disp': 0
            }
            es = cma.CMAEvolutionStrategy(x_init, 0.1, opts)
            es.optimize(lambda x: objective(x, n, m, beta, lambda_, gamma))
            x_cmaes = es.result.xbest
            A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes = unflatten_params(x_cmaes, n, m)
            obj_cmaes, recon_ll_cmaes, recon_l2_cmaes, pairwise_mi_cmaes = gamma_1(A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes, beta, lambda_, gamma)
            mi_enc_cmaes = encoder_mi(B_cmaes, Sigma_W_cmaes)
            mi_dec_cmaes = decoder_mi(A_cmaes, Sigma_Z_cmaes)
            sub_cov_XV_cmaes = sub_covariance_XV(B_cmaes)
            mi_matrix_cmaes = compute_mutual_information_matrix(B_cmaes, Sigma_W_cmaes)
            I5_cmaes = disentanglement_I5(B_cmaes, Sigma_W_cmaes)

            # Store reconstruction errors for 3D plots
            recon_ll_data['lbfgs'][beta_idx, lambda_idx, gamma_idx] = recon_ll_lbfgs if not np.isinf(recon_ll_lbfgs) else np.nan
            recon_ll_data['cmaes'][beta_idx, lambda_idx, gamma_idx] = recon_ll_cmaes if not np.isinf(recon_ll_cmaes) else np.nan
            recon_l2_data['lbfgs'][beta_idx, lambda_idx, gamma_idx] = recon_l2_lbfgs if not np.isinf(recon_l2_lbfgs) else np.nan
            recon_l2_data['cmaes'][beta_idx, lambda_idx, gamma_idx] = recon_l2_cmaes if not np.isinf(recon_l2_cmaes) else np.nan

            # Generate and save mutual information heatmaps for each method
            for method, mi_matrix in [
                ('lbfgs', mi_matrix_lbfgs),
                ('cmaes', mi_matrix_cmaes)
            ]:
                plt.figure(figsize=(8, 6))
                if np.all(np.isfinite(mi_matrix)):
                    sns.heatmap(
                        mi_matrix,
                        annot=True, fmt=".2f", cmap='Blues',  # Blueish colormap
                        xticklabels=[r'$v_{' + str(i+1) + '}$' for i in range(s)],
                        yticklabels=[r'$x_{' + str(i+1) + '}$' for i in range(m)],
                        cbar_kws={'label': 'Mutual Information (nats)'},
                        annot_kws={"size": 16}  # Increased font size for annotations
                    )
                    plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, γ={gamma}, Method={method})")
                    plt.xlabel("Generative Variables")
                    plt.ylabel("Latent Variables")
                else:
                    plt.text(0.5, 0.5, "Invalid Data (contains inf or nan)",
                             horizontalalignment='center', verticalalignment='center')
                    plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, γ={gamma}, Method={method})")
                    plt.xlabel("Generative Variables")
                    plt.ylabel("Latent Variables")
                plt.savefig(os.path.join(output_dir, f"heatmap_beta{beta}_lambda{lambda_}_gamma{gamma}_{method}.png"))
                plt.close()

            # Store results
            result = {
                'beta': beta,
                'lambda': lambda_,
                'gamma': gamma,
                'lbfgs': {
                    'obj': obj_lbfgs,
                    'recon_ll': recon_ll_lbfgs,
                    'recon_l2': recon_l2_lbfgs,
                    'mi_enc': mi_enc_lbfgs,
                    'mi_dec': mi_dec_lbfgs,
                    'pairwise_mi': pairwise_mi_lbfgs,
                    'sub_cov_XV': sub_cov_XV_lbfgs,
                    'I5': I5_lbfgs,
                    'mi_matrix': mi_matrix_lbfgs
                },
                'cmaes': {
                    'obj': obj_cmaes,
                    'recon_ll': recon_ll_cmaes,
                    'recon_l2': recon_l2_cmaes,
                    'mi_enc': mi_enc_cmaes,
                    'mi_dec': mi_dec_cmaes,
                    'pairwise_mi': pairwise_mi_cmaes,
                    'sub_cov_XV': sub_cov_XV_cmaes,
                    'I5': I5_cmaes,
                    'mi_matrix': mi_matrix_cmaes
                }
            }
            results.append(result)

# Generate 3D surface plots for reconstruction errors
for method in ['lbfgs', 'cmaes']:
    # Prepare data for plotting
    beta_grid, lambda_grid, gamma_grid = np.meshgrid(beta_values, lambda_values, gamma_values, indexing='ij')
    
    # Log-Likelihood Reconstruction Error - Vary beta and lambda, average over gamma
    recon_ll_avg_gamma = np.nanmean(recon_ll_data[method], axis=2)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(beta_grid[:, :, 0], lambda_grid[:, :, 0], recon_ll_avg_gamma, cmap='Reds')
    ax.set_xlabel('β')
    ax.set_ylabel('λ')
    ax.set_zlabel('Log-Likelihood Reconstruction Error')
    ax.set_title(f'Log-Likelihood Error (Method={method}, Averaged over γ)')
    fig.colorbar(surf, ax=ax, label='Log-Likelihood Reconstruction Error')
    plt.savefig(os.path.join(output_dir, f"3d_recon_ll_avg_gamma_{method}.png"))
    plt.close()

    # L2-Norm Reconstruction Error - Vary beta and lambda, average over gamma
    recon_l2_avg_gamma = np.nanmean(recon_l2_data[method], axis=2)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(beta_grid[:, :, 0], lambda_grid[:, :, 0], recon_l2_avg_gamma, cmap='Reds')
    ax.set_xlabel('β')
    ax.set_ylabel('λ')
    ax.set_zlabel('L2-Norm Reconstruction Error')
    ax.set_title(f'L2-Norm Error (Method={method}, Averaged over γ)')
    fig.colorbar(surf, ax=ax, label='L2-Norm Reconstruction Error')
    plt.savefig(os.path.join(output_dir, f"3d_recon_l2_avg_gamma_{method}.png"))
    plt.close()

# Print results with 2 decimal places
print("\nFinal Results:")
for result in results:
    beta = result['beta']
    lambda_ = result['lambda']
    gamma = result['gamma']
    print(f"\nBeta={beta}, Lambda={lambda_}, Gamma={gamma}")
    print(f"{'Method':<15} {'Objective':>12} {'Log-Likelihood':>15} {'L2-Norm':>12} {'Encoder MI':>12} {'Decoder MI':>12} {'I5':>12} {'Pairwise MI':>15}")
    print("-" * 110)
    for method in ['lbfgs', 'cmaes']:
        data = result[method]
        obj = data['obj']
        recon_ll = data['recon_ll']
        recon_l2 = data['recon_l2']
        mi_enc = data['mi_enc']
        mi_dec = data['mi_dec']
        I5 = data['I5']
        pairwise_mi = data['pairwise_mi']
        print(f"{method.replace('_', ' ').title():<15} {obj:>12.2f} {recon_ll:>15.2f} {recon_l2:>12.2f} {mi_enc:>12.2f} {mi_dec:>12.2f} {I5:>12.2f} {pairwise_mi:>15.2f}")


Testing beta=6, lambda=1, gamma=4

Final Results:

Beta=6, Lambda=1, Gamma=4
Method             Objective  Log-Likelihood      L2-Norm   Encoder MI   Decoder MI           I5     Pairwise MI
--------------------------------------------------------------------------------------------------------------
Lbfgs                 514.06           65.00       227.49         3.61         3.18         1.48            0.61
Cmaes                 632.46          105.37       271.57         3.57         3.43         1.70            0.34


In [5]:
import numpy as np
import plotly.graph_objects as go
import os

# Create the output folder on the Desktop
desktop_path = os.path.expanduser("~/Desktop")
output_dir = os.path.join(desktop_path, "lambda gamma beta VAE")
os.makedirs(output_dir, exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Define the hyperparameter values
beta_values = [1, 2, 4, 6]  # 4 beta values
lambda_values = [1, 2, 4, 6]  # 4 lambda values
gamma_values = [1, 2, 4, 6]  # 4 gamma values

# Randomize reconstruction errors for each tuple
# Log-likelihood errors (negative, e.g., -100 to -50)
# L2-norm errors (positive, e.g., 20 to 40)
recon_ll_data = {
    'lbfgs': np.random.uniform(low=-100, high=-50, size=(len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.random.uniform(low=-100, high=-50, size=(len(beta_values), len(lambda_values), len(gamma_values)))
}
recon_l2_data = {
    'lbfgs': np.random.uniform(low=20, high=40, size=(len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.random.uniform(low=20, high=40, size=(len(beta_values), len(lambda_values), len(gamma_values)))
}

# Function to create and save an interactive 3D surface plot for a fixed gamma
def plot_3d_surface_for_gamma(method, error_type, gamma, gamma_idx):
    # Prepare grids for surface plot
    beta_grid, lambda_grid = np.meshgrid(beta_values, lambda_values)
    
    # Select data based on error type
    data = recon_ll_data[method] if error_type == 'log_likelihood' else recon_l2_data[method]
    error_label = 'Log-Likelihood Reconstruction Error' if error_type == 'log_likelihood' else 'L2-Norm Reconstruction Error'
    
    # Get the reconstruction error for this gamma
    Z = data[:, :, gamma_idx]  # Shape: (4, 4) for fixed gamma
    
    # Create a surface trace
    surface = go.Surface(
        x=beta_grid,
        y=lambda_grid,
        z=Z,
        surfacecolor=Z,  # Use z-values (error) to determine color
        colorscale='Reds',
        opacity=0.8,
        colorbar=dict(title=error_label),
        hovertemplate='β: %{x}<br>λ: %{y}<br>γ: ' + str(gamma) + '<br>' + error_label + ': %{z}<br>'
    )
    
    # Create the layout for the plot
    layout = go.Layout(
        title=f'{error_label} for γ={gamma} (Method={method})',
        scene=dict(
            xaxis_title='β',
            yaxis_title='λ',
            zaxis_title=error_label,
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        margin=dict(l=65, r=50, b=65, t=90)
    )
    
    # Create the figure
    fig = go.Figure(data=[surface], layout=layout)
    
    # Save the plot as an interactive HTML file
    output_file = os.path.join(output_dir, f"3d_surface_{error_label.replace(' ', '_').lower()}_gamma{gamma}_{method}.html")
    fig.write_html(output_file)
    print(f"Saved interactive plot to {output_file}")

# Create and save interactive plots for all combinations
methods = ['lbfgs', 'cmaes']
error_types = ['log_likelihood', 'l2_norm']

# Loop over all methods, error types, and gamma values
for method in methods:
    for error_type in error_types:
        for gamma_idx, gamma in enumerate(gamma_values):
            plot_3d_surface_for_gamma(method, error_type, gamma, gamma_idx)

Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_log-likelihood_reconstruction_error_gamma1_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_log-likelihood_reconstruction_error_gamma2_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_log-likelihood_reconstruction_error_gamma4_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_log-likelihood_reconstruction_error_gamma6_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_l2-norm_reconstruction_error_gamma1_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_l2-norm_reconstruction_error_gamma2_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lambda gamma beta VAE/3d_surface_l2-norm_reconstruction_error_gamma4_lbfgs.html
Saved interactive plot to /Users/minitechy/Desktop/lamb

In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import Bounds
import cma
from numpy.linalg import det
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import permutations
import plotly.graph_objects as go

# Create the output folder on the Desktop
desktop_path = os.path.expanduser("~/Desktop")
output_dir = os.path.join(desktop_path, "lambda_gamma_beta_VAE_linear_results")
os.makedirs(output_dir, exist_ok=True)

# Enable LaTeX rendering for labels (usetex=False to avoid requiring LaTeX installation)
plt.rc('text', usetex=False)

# Set random seed for reproducibility
np.random.seed(42)

# Dimensions and hyperparameters
n = 20  # Data dimension
m = 5   # Latent dimension
s = 5   # Number of generative variables
max_iter = 2000   # Max iterations for optimization
beta_values = [1, 2, 4, 6]  # 4 beta values
lambda_values = [1, 2, 4, 6]  # 4 lambda values
gamma_values = [1, 2, 4, 6]  # 4 gamma values
sigma2 = 0.1  # Noise variance for Sigma_Z_tilde

# Generate Sigma_V (5x5 diagonal matrix for 5 independent generative variables)
sigma_v = 1 + 4 * np.random.rand(s)  # Standard deviations between 1 and 5
Sigma_V = np.diag(sigma_v**2)  # Diagonal matrix of variances

# Generate Gamma (n x s matrix)
Gamma = np.random.randn(n, s)  # Random transformation matrix

# Compute Sigma_Y using the generative model
Sigma_Y = Gamma @ Sigma_V @ Gamma.T + sigma2 * np.eye(n)
# Ensure symmetry and positive definiteness
Sigma_Y = (Sigma_Y + Sigma_Y.T) / 2
eigvals = np.linalg.eigvalsh(Sigma_Y)
if np.any(eigvals <= 0):
    Sigma_Y += (1e-6 - np.min(eigvals)) * np.eye(n)  # Adjust for positive definiteness
Sigma_Y_inv = np.linalg.inv(Sigma_Y)

# Sub-covariance matrix XV
def sub_covariance_XV(B):
    try:
        return B @ Gamma @ Sigma_V  # Shape: (m x s) = (5 x 5)
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Compute mutual information matrix for all pairs (x_i, v_j)
def compute_mutual_information_matrix(B, Sigma_W):
    try:
        # Compute covariance of X: B * Sigma_Y * B.T + Sigma_W
        Sigma_X = B @ Sigma_Y @ B.T + Sigma_W
        if det(Sigma_X) <= 0:
            return np.full((m, s), np.inf)
        
        # Sub-covariance matrix: cov(X, V) = B * Gamma * Sigma_V
        cov_XV = sub_covariance_XV(B)
        if np.any(np.isinf(cov_XV)):
            return np.full((m, s), np.inf)
        
        # Initialize mutual information matrix
        mi_matrix = np.zeros((m, s))
        for i in range(m):
            for j in range(s):
                Sigma_xi = Sigma_X[i, i]  # Variance of x_i
                Sigma_vj = Sigma_V[j, j]  # Variance of v_j
                cov_xi_vj = cov_XV[i, j]  # cov(x_i, v_j)
                Sigma_xi_vj = np.array([[Sigma_xi, cov_xi_vj],
                                       [cov_xi_vj, Sigma_vj]])
                det_xi_vj = det(Sigma_xi_vj)
                if det_xi_vj <= 0:
                    mi_matrix[i, j] = np.inf
                else:
                    mi_matrix[i, j] = 0.5 * np.log(Sigma_xi * Sigma_vj / det_xi_vj)
        return mi_matrix
    except np.linalg.LinAlgError:
        return np.full((m, s), np.inf)

# Disentanglement metric I_5 (maximum sum over all permutations)
def disentanglement_I5(B, Sigma_W):
    try:
        # Compute the mutual information matrix
        mi_matrix = compute_mutual_information_matrix(B, Sigma_W)
        if np.any(np.isinf(mi_matrix)):
            return np.inf
        
        # Compute I_5 as the maximum sum across all permutations
        max_sum = -np.inf
        for perm in permutations(range(s)):
            current_sum = sum(mi_matrix[i, pi] for i, pi in enumerate(perm))
            max_sum = max(max_sum, current_sum)
        
        return max_sum
    except np.linalg.LinAlgError:
        return np.inf

# Mutual information functions
def encoder_mi(B, Sigma_W):
    try:
        num = B @ Sigma_Y @ B.T + Sigma_W
        det_num = det(num)
        det_den = det(Sigma_W)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

def decoder_mi(A, Sigma_Z):
    try:
        num = A @ A.T + Sigma_Z
        det_num = det(num)
        det_den = det(Sigma_Z)
        if det_num <= 0 or det_den <= 0:
            return np.inf
        return 0.5 * np.log(det_num / det_den)
    except np.linalg.LinAlgError:
        return np.inf

# Compute sum of I(X_i; \tilde{X}_i) for each i
def compute_pairwise_mi(B, Sigma_W, A, Sigma_Z):
    try:
        # Compute cov(X) = B * Sigma_Y * B^T + Sigma_W
        Sigma_X = B @ Sigma_Y @ B.T + Sigma_W
        
        # Compute cov(\tilde{X}) = B * A * A^T * B^T + B * Sigma_Z * B^T + Sigma_W
        term1 = B @ (A @ A.T) @ B.T  # B * A * A^T * B^T
        term2 = B @ Sigma_Z @ B.T     # B * Sigma_Z * B^T
        term3 = Sigma_W               # Sigma_W
        Sigma_X_tilde = term1 + term2 + term3
        
        # Initialize sum of I(X_i; \tilde{X}_i)
        mi_sum = 0.0
        for i in range(m):
            # Extract diagonal elements
            sigma_w_ii = Sigma_W[i, i]         # [Sigma_W]_{ii}
            sigma_x_ii = Sigma_X[i, i]         # [B * Sigma_Y * B^T + Sigma_W]_{ii}
            sigma_x_tilde_ii = Sigma_X_tilde[i, i]  # [B * A * A^T * B^T + B * Sigma_Z * B^T + Sigma_W]_{ii}
            
            # Compute the fraction inside the log
            fraction = (sigma_w_ii**2) / (sigma_x_ii * sigma_x_tilde_ii)
            if fraction >= 1 or fraction <= 0:  # Ensure valid log argument
                return np.inf
            
            # Compute I(X_i; \tilde{X}_i) = -0.5 * log(1 - fraction)
            mi_i = -0.5 * np.log(1 - fraction)
            mi_sum += mi_i
        
        return mi_sum  # Return sum for objective function and reporting
    except (np.linalg.LinAlgError, ValueError):
        return np.inf

# Objective function Gamma_1 with reconstruction errors, modified for lambda gamma beta-VAE
def gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_, gamma):
    try:
        Sigma_Z_inv = np.linalg.inv(Sigma_Z)
        Sigma_W_inv = np.linalg.inv(Sigma_W)
        
        # Check determinants for positive definiteness
        det_Sigma_Z = det(Sigma_Z)
        det_Sigma_W = det(Sigma_W)
        if det_Sigma_Z <= 0 or det_Sigma_W <= 0:
            return np.inf, np.inf, np.inf, np.inf
        
        # First term: Log-likelihood reconstruction error
        trace1 = np.trace(A.T @ Sigma_Z_inv @ Sigma_Y @ B.T)
        trace2 = np.trace(Sigma_Z_inv @ A @ B @ Sigma_Y)
        trace3 = np.trace(Sigma_Z_inv @ Sigma_Y)
        trace4 = np.trace(A.T @ Sigma_Z_inv @ A @ (B @ Sigma_Y @ B.T + Sigma_W))
        recon_ll = -0.5 * (trace1 + trace2 - trace3 - trace4 - \
                           n * np.log(2 * np.pi) - np.log(det_Sigma_Z))
        
        # Second term: KL divergence
        term2 = 0.5 * beta * (np.trace(B @ Sigma_Y @ B.T + Sigma_W) - \
                              np.log(det_Sigma_W) - m)
        
        # Third term: L2-norm with lambda for objective
        l2_base = np.trace(
            (np.eye(n) - A @ B) @ Sigma_Y @ (np.eye(n) - A @ B).T + \
            A @ Sigma_W @ A.T
        )
        term3 = lambda_ * l2_base
        recon_l2 = l2_base  # Unscaled L2-norm for reporting
        
        # Compute sum of I(X_i; \tilde{X}_i)
        pairwise_mi_sum = compute_pairwise_mi(B, Sigma_W, A, Sigma_Z)
        if np.isinf(pairwise_mi_sum):
            return np.inf, np.inf, np.inf, np.inf
        
        # New objective: Gamma_1 - gamma * sum I(X_i; \tilde{X}_i)
        gamma_term = gamma * pairwise_mi_sum
        total = recon_ll + term2 + term3 - gamma_term
        
        return total, recon_ll, recon_l2, pairwise_mi_sum
    except np.linalg.LinAlgError:
        return np.inf, np.inf, np.inf, np.inf

# Flatten parameters
def flatten_params(A, B, Sigma_Z, Sigma_W):
    L_W = np.linalg.cholesky(Sigma_W)
    L_W_vec = L_W[np.tril_indices(m)]
    return np.concatenate([
        A.flatten(),
        B.flatten(),
        np.diag(Sigma_Z).flatten(),
        L_W_vec.flatten()
    ])

# Unflatten parameters
def unflatten_params(x, n, m):
    idx_A = n * m
    idx_B = idx_A + m * n
    idx_Sigma_Z = idx_B + n
    A = x[:idx_A].reshape(n, m)
    B = x[idx_A:idx_B].reshape(m, n)
    diag_Z = x[idx_B:idx_Sigma_Z]
    Sigma_Z = np.diag(np.maximum(diag_Z, 1e-1))
    L_W_vec = x[idx_Sigma_Z:]
    L_W = np.zeros((m, m))
    tril_indices = np.tril_indices(m)
    L_W[tril_indices] = L_W_vec
    for i in range(m):
        L_W[i, i] = np.maximum(L_W[i, i], 2)
    Sigma_W = L_W @ L_W.T
    return A, B, Sigma_Z, Sigma_W

# Objective function for L-BFGS-B and CMA-ES
def objective(x, n, m, beta, lambda_, gamma):
    A, B, Sigma_Z, Sigma_W = unflatten_params(x, n, m)
    total, _, _, _ = gamma_1(A, B, Sigma_Z, Sigma_W, beta, lambda_, gamma)
    return total

# Function to create and save an interactive 3D surface plot for a fixed gamma
def plot_3d_surface_for_gamma(method, error_type, gamma, gamma_idx, data):
    # Prepare grids for surface plot
    beta_grid, lambda_grid = np.meshgrid(beta_values, lambda_values)
    
    # Select data based on error type
    error_label = 'Log-Likelihood Reconstruction Error' if error_type == 'log_likelihood' else 'L2-Norm Reconstruction Error'
    
    # Get the reconstruction error for this gamma
    Z = data[method][:, :, gamma_idx]  # Shape: (4, 4) for fixed gamma
    
    # Create a surface trace
    surface = go.Surface(
        x=beta_grid,
        y=lambda_grid,
        z=Z,
        surfacecolor=Z,  # Use z-values (error) to determine color
        colorscale='Reds',
        opacity=0.8,
        colorbar=dict(title=error_label),
        hovertemplate='β: %{x}<br>λ: %{y}<br>γ: ' + str(gamma) + '<br>' + error_label + ': %{z}<br>'
    )
    
    # Create the layout for the plot
    layout = go.Layout(
        title=f'{error_label} for γ={gamma} (Method={method})',
        scene=dict(
            xaxis_title='β',
            yaxis_title='λ',
            zaxis_title=error_label,
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        margin=dict(l=65, r=50, b=65, t=90)
    )
    
    # Create the figure
    fig = go.Figure(data=[surface], layout=layout)
    
    # Save the plot as an interactive HTML file
    output_file = os.path.join(output_dir, f"3d_surface_{error_label.replace(' ', '_').lower()}_gamma{gamma}_{method}.html")
    fig.write_html(output_file)
    print(f"Saved interactive plot to {output_file}")

# Initial parameters
A_init = 0.1 * np.random.randn(n, m)
B_init = 0.1 * np.random.randn(m, n)
Sigma_Z_init = np.diag(1 + 0.01 * np.abs(np.random.randn(n)))
Sigma_W_init = np.eye(m) + 0.01 * np.random.randn(m, m)
Sigma_W_init = (Sigma_W_init + Sigma_W_init.T) / 2 + 1e-1 * np.eye(m)

# Store results and reconstruction errors for each method
results = []
# Dictionaries to store reconstruction errors for 3D plots
recon_ll_data = {
    'lbfgs': np.zeros((len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.zeros((len(beta_values), len(lambda_values), len(gamma_values)))
}
recon_l2_data = {
    'lbfgs': np.zeros((len(beta_values), len(lambda_values), len(gamma_values))),
    'cmaes': np.zeros((len(beta_values), len(lambda_values), len(gamma_values)))
}

# Loop over beta, lambda, and gamma values
for beta_idx, beta in enumerate(beta_values):
    for lambda_idx, lambda_ in enumerate(lambda_values):
        for gamma_idx, gamma in enumerate(gamma_values):
            print(f"\nTesting beta={beta}, lambda={lambda_}, gamma={gamma}")

            # L-BFGS-B Optimization
            x_init = flatten_params(A_init, B_init, Sigma_Z_init, Sigma_W_init)
            lb = np.concatenate([
                -10 * np.ones(n * m),
                -10 * np.ones(m * n),
                1e-1 * np.ones(n),
                -10 * np.ones((m * (m + 1)) // 2)
            ])
            ub = np.concatenate([
                10 * np.ones(n * m),
                10 * np.ones(m * n),
                np.inf * np.ones(n),
                10 * np.ones((m * (m + 1)) // 2)
            ])
            bounds = Bounds(lb, ub)
            result_lbfgs = minimize(
                lambda x: objective(x, n, m, beta, lambda_, gamma), x_init,
                method='L-BFGS-B', bounds=bounds,
                options={'disp': False, 'maxiter': 2000, 'gtol': 1e-5}
            )
            A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs = unflatten_params(result_lbfgs.x, n, m)
            obj_lbfgs, recon_ll_lbfgs, recon_l2_lbfgs, pairwise_mi_lbfgs = gamma_1(A_lbfgs, B_lbfgs, Sigma_Z_lbfgs, Sigma_W_lbfgs, beta, lambda_, gamma)
            mi_enc_lbfgs = encoder_mi(B_lbfgs, Sigma_W_lbfgs)
            mi_dec_lbfgs = decoder_mi(A_lbfgs, Sigma_Z_lbfgs)
            sub_cov_XV_lbfgs = sub_covariance_XV(B_lbfgs)
            mi_matrix_lbfgs = compute_mutual_information_matrix(B_lbfgs, Sigma_W_lbfgs)
            I5_lbfgs = disentanglement_I5(B_lbfgs, Sigma_W_lbfgs)

            # CMA-ES Optimization
            opts = {
                'bounds': [lb, ub],
                'tolfun': 1e-8,
                'maxiter': 2000,
                'maxfevals': 50000,
                'popsize': 100,
                'verb_disp': 0
            }
            es = cma.CMAEvolutionStrategy(x_init, 0.1, opts)
            es.optimize(lambda x: objective(x, n, m, beta, lambda_, gamma))
            x_cmaes = es.result.xbest
            A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes = unflatten_params(x_cmaes, n, m)
            obj_cmaes, recon_ll_cmaes, recon_l2_cmaes, pairwise_mi_cmaes = gamma_1(A_cmaes, B_cmaes, Sigma_Z_cmaes, Sigma_W_cmaes, beta, lambda_, gamma)
            mi_enc_cmaes = encoder_mi(B_cmaes, Sigma_W_cmaes)
            mi_dec_cmaes = decoder_mi(A_cmaes, Sigma_Z_cmaes)
            sub_cov_XV_cmaes = sub_covariance_XV(B_cmaes)
            mi_matrix_cmaes = compute_mutual_information_matrix(B_cmaes, Sigma_W_cmaes)
            I5_cmaes = disentanglement_I5(B_cmaes, Sigma_W_cmaes)

            # Store reconstruction errors for 3D plots
            recon_ll_data['lbfgs'][beta_idx, lambda_idx, gamma_idx] = recon_ll_lbfgs if not np.isinf(recon_ll_lbfgs) else np.nan
            recon_ll_data['cmaes'][beta_idx, lambda_idx, gamma_idx] = recon_ll_cmaes if not np.isinf(recon_ll_cmaes) else np.nan
            recon_l2_data['lbfgs'][beta_idx, lambda_idx, gamma_idx] = recon_l2_lbfgs if not np.isinf(recon_l2_lbfgs) else np.nan
            recon_l2_data['cmaes'][beta_idx, lambda_idx, gamma_idx] = recon_l2_cmaes if not np.isinf(recon_l2_cmaes) else np.nan

            # Generate and save mutual information heatmaps for each method
            for method, mi_matrix in [
                ('lbfgs', mi_matrix_lbfgs),
                ('cmaes', mi_matrix_cmaes)
            ]:
                plt.figure(figsize=(8, 6))
                if np.all(np.isfinite(mi_matrix)):
                    sns.heatmap(
                        mi_matrix,
                        annot=True, fmt=".2f", cmap='Blues',  # Blueish colormap
                        xticklabels=[r'$v_{' + str(i+1) + '}$' for i in range(s)],
                        yticklabels=[r'$x_{' + str(i+1) + '}$' for i in range(m)],
                        cbar_kws={'label': 'Mutual Information (nats)'},
                        annot_kws={"size": 16}  # Increased font size for annotations
                    )
                    plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, γ={gamma}, Method={method})")
                    plt.xlabel("Generative Variables")
                    plt.ylabel("Latent Variables")
                else:
                    plt.text(0.5, 0.5, "Invalid Data (contains inf or nan)",
                             horizontalalignment='center', verticalalignment='center')
                    plt.title(f"Mutual Information Heatmap (β={beta}, λ={lambda_}, γ={gamma}, Method={method})")
                    plt.xlabel("Generative Variables")
                    plt.ylabel("Latent Variables")
                plt.savefig(os.path.join(output_dir, f"heatmap_beta{beta}_lambda{lambda_}_gamma{gamma}_{method}.png"))
                plt.close()

            # Store results
            result = {
                'beta': beta,
                'lambda': lambda_,
                'gamma': gamma,
                'lbfgs': {
                    'obj': obj_lbfgs,
                    'recon_ll': recon_ll_lbfgs,
                    'recon_l2': recon_l2_lbfgs,
                    'mi_enc': mi_enc_lbfgs,
                    'mi_dec': mi_dec_lbfgs,
                    'pairwise_mi': pairwise_mi_lbfgs,
                    'sub_cov_XV': sub_cov_XV_lbfgs,
                    'I5': I5_lbfgs,
                    'mi_matrix': mi_matrix_lbfgs
                },
                'cmaes': {
                    'obj': obj_cmaes,
                    'recon_ll': recon_ll_cmaes,
                    'recon_l2': recon_l2_cmaes,
                    'mi_enc': mi_enc_cmaes,
                    'mi_dec': mi_dec_cmaes,
                    'pairwise_mi': pairwise_mi_cmaes,
                    'sub_cov_XV': sub_cov_XV_cmaes,
                    'I5': I5_cmaes,
                    'mi_matrix': mi_matrix_cmaes
                }
            }
            results.append(result)

# Generate interactive 3D surface plots for reconstruction errors using Plotly
methods = ['lbfgs', 'cmaes']
error_types = ['log_likelihood', 'l2_norm']

# Loop over all methods, error types, and gamma values to create plots
for method in methods:
    for error_type in error_types:
        data = recon_ll_data if error_type == 'log_likelihood' else recon_l2_data
        for gamma_idx, gamma in enumerate(gamma_values):
            plot_3d_surface_for_gamma(method, error_type, gamma, gamma_idx, data)

# Print results with 2 decimal places
print("\nFinal Results:")
for result in results:
    beta = result['beta']
    lambda_ = result['lambda']
    gamma = result['gamma']
    print(f"\nBeta={beta}, Lambda={lambda_}, Gamma={gamma}")
    print(f"{'Method':<15} {'Objective':>12} {'Log-Likelihood':>15} {'L2-Norm':>12} {'Encoder MI':>12} {'Decoder MI':>12} {'I5':>12} {'Pairwise MI':>15}")
    print("-" * 110)
    for method in ['lbfgs', 'cmaes']:
        data = result[method]
        obj = data['obj']
        recon_ll = data['recon_ll']
        recon_l2 = data['recon_l2']
        mi_enc = data['mi_enc']
        mi_dec = data['mi_dec']
        I5 = data['I5']
        pairwise_mi = data['pairwise_mi']
        print(f"{method.replace('_', ' ').title():<15} {obj:>12.2f} {recon_ll:>15.2f} {recon_l2:>12.2f} {mi_enc:>12.2f} {mi_dec:>12.2f} {I5:>12.2f} {pairwise_mi:>15.2f}")


Testing beta=1, lambda=1, gamma=1

Testing beta=1, lambda=1, gamma=2

Testing beta=1, lambda=1, gamma=4

Testing beta=1, lambda=1, gamma=6

Testing beta=1, lambda=2, gamma=1

Testing beta=1, lambda=2, gamma=2

Testing beta=1, lambda=2, gamma=4

Testing beta=1, lambda=2, gamma=6

Testing beta=1, lambda=4, gamma=1

Testing beta=1, lambda=4, gamma=2

Testing beta=1, lambda=4, gamma=4

Testing beta=1, lambda=4, gamma=6

Testing beta=1, lambda=6, gamma=1

Testing beta=1, lambda=6, gamma=2

Testing beta=1, lambda=6, gamma=4

Testing beta=1, lambda=6, gamma=6

Testing beta=2, lambda=1, gamma=1

Testing beta=2, lambda=1, gamma=2

Testing beta=2, lambda=1, gamma=4

Testing beta=2, lambda=1, gamma=6

Testing beta=2, lambda=2, gamma=1

Testing beta=2, lambda=2, gamma=2

Testing beta=2, lambda=2, gamma=4

Testing beta=2, lambda=2, gamma=6

Testing beta=2, lambda=4, gamma=1

Testing beta=2, lambda=4, gamma=2

Testing beta=2, lambda=4, gamma=4

Testing beta=2, lambda=4, gamma=6

Testing beta=2, lam